<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day05_practice3_k%EA%B2%B9_%EA%B5%90%EC%B0%A8%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 성능 검증 - k겹 교차검증과 드롭아웃
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib as plt
import numpy as np

plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 셀 1. 데이터 불러오기
CSV_URL  = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/sonar3.csv"
CSV_PATH = "sonar3.csv"

def load_sonar():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH, header=None) # 첫 행부터 바로 숫자 데이터라서 header=None이 필수

  try:
    df = pd.read_csv(CSV_URL, header=None)
    df.to_csv(CSV_PATH, index=False, header=False)
    print(f"다운로드 완료 -> {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_sonar()
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values # 라벨 (0=암석, 1=금속)

print(f"데이터: {X.shape[0]}개 x 특징 {X.shape[1]}개")
print(f"클래스: 암석 {(y==0).sum()}개 / 금속 {(y==1).sum()}개")

로컬 파일 재사용: sonar3.csv
데이터: 208개 x 특징 60개
클래스: 암석 97개 / 금속 111개


In [3]:
# 셀 2. 학습 - 평가 함수
def make_model(dropout=0.0):
  return nn.Sequential(
      nn.Linear(60, 128), nn.ReLU(), nn.Dropout(dropout), # 활성화를 통과한 뉴런 출력 중 일부를 끄는 것
      nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
      nn.Linear(64, 1), nn.Sigmoid(), # 0~1 사이의 확률로 변환
  )

def train_eval(X_tr, y_tr, X_te, y_te, dropout=0.0, epochs=200):
  scaler = StandardScaler()
  X_tr = scaler.fit_transform(X_tr)
  X_te = scaler.transform(X_te)

  X_tr = torch.tensor(X_tr, dtype=torch.float32).to(device)
  y_tr = torch.tensor(y_tr, dtype=torch.float32).reshape(-1,1).to(device)
  X_te = torch.tensor(X_te, dtype=torch.float32).to(device)
  y_te = torch.tensor(y_te, dtype=torch.float32).reshape(-1,1).to(device)

  model = make_model(dropout).to(device)
  loss_fn = nn.BCELoss()
  cpt = torch.optim.Adam(model.parameters(), lr=0.005)

  model.train() # 드롭아웃 ON
  for _ in range(epochs):
    loss = loss_fn(model(X_tr), y_tr)
    cpt.zero_grad()
    loss.backward()
    cpt.step()

  model.eval() # 드롭아웃 OFF
  with torch.no_grad():
    train_acc = ((model(X_tr) > 0.5) == y_tr.bool()).float().mean().item()
    test_acc = ((model(X_te) > 0.5) == y_te.bool()).float().mean().item()

  return train_acc, test_acc

In [4]:
# 셀 4. k겹 교차검증 - 모두가 한 번씩 시험지가 된다
print("\n5겹 교차검증:")

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)  # 나눌 때 클래스 비율까지 유지

fold_accs = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y)): # 겹 번호를 1부터, kfold.split(X,y)는 데이터를 K번 나누며, 매 반복마다
    torch.manual_seed(42)

    _, acc = train_eval(
        X[train_idx], y[train_idx],
        X[test_idx], y[test_idx],
        epochs=100
    )

    fold_accs.append(acc)
    print(f"{fold + 1}번째 겹 -> 정확도 {acc:.3f}")

mean, std = np.mean(fold_accs), np.std(fold_accs) # 평균 ± 표준편차 (값이 크면 데이터에 따라 들쑥날쑥한 불안한 모델)

print(f"\n최종 성적표: {mean:.3f} ± {std:.3f}")


5겹 교차검증:
1번째 겹 -> 정확도 0.857
2번째 겹 -> 정확도 0.905
3번째 겹 -> 정확도 0.857
4번째 겹 -> 정확도 0.854
5번째 겹 -> 정확도 0.854

최종 성적표: 0.865 ± 0.020


In [6]:
# 셀 5. 실험 3 - 과적합 진단
# 드롭아웃: 학습 떄 뉴런을 무작위로 잠시 꺼서(p=30%) '특정 뉴런 의존'을 막는 과적합 대책
# train 정확도까지 같이 봐야 과적합이 보인다.

print("\n[실험 3] 드롭아웃 없음 vs 드롭아웃 30% (같은 5겹, train/test 동시 측정):")
for dr in [0.0, 0.3]:
  tr_accs, te_accs = [], []

  for train_idx, test_idx in kfold.split(X, y):
    torch.manual_seed(42)

    tr, te = train_eval(
        X[train_idx], y[train_idx],
        X[test_idx], y[test_idx],
        dropout=dr,
        epochs=100
    )

    tr_accs.append(tr)
    te_accs.append(te)

  label = "없음(0.0)" if dr == 0.0 else "있음(0.3)"
  gap = np.mean(tr_accs) - np.mean(te_accs)

  print(f" 드롭아웃 {label} -> train {np.mean(tr_accs):.3f} | "
        f"test {np.mean(te_accs):.3f} ± {np.std(te_accs):.3f} | 격차 {gap:+.3f}")

  # Sonar 데이터셋의 특성 - 표본이 작고 특성이 없고 이진 분류이다.
  # 에폭을 크게 줄인다 (100~200), train이 1.000에 닿기 전 구간에서 비교
  # 드롭아웃을 더 강하게 본다 (0.5까지)
  # 은닉층을 줄인다 (128->64, 64->32)


[실험 3] 드롭아웃 없음 vs 드롭아웃 30% (같은 5겹, train/test 동시 측정):
 드롭아웃 없음(0.0) -> train 1.000 | test 0.865 ± 0.020 | 격차 +0.135
 드롭아웃 있음(0.3) -> train 1.000 | test 0.837 ± 0.022 | 격차 +0.163
